<div class="alert alert-block alert-info">
    
## Imports & Parameters
</div>

In [1]:
## Import
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [2]:
## Parameter
start_date = '20240915'
end_date = '20240930'
refresh_month = 'sep'
city = 'Rest'

In [3]:
# Get the current working directory
cwd = os.getcwd()
local_extract = '/Users/E2074/local-datasets/customer/appography/datasets/' + city.lower() + '/local_extract/'

print(cwd)
print(local_extract)

/Users/E2074/analytics/customer/appography/age_classification_notebook/simplified-version
/Users/E2074/local-datasets/customer/appography/datasets/rest/local_extract/


<div class="alert alert-block alert-info">
    
## Datasets
</div>

<div class="alert alert-block alert-warning">

### Metric Data
</div>

In [5]:
print('Query 1 running..')
start_time = time.time()

active_customer_base = f"""

    WITH fe_tbl AS (
    
        SELECT 
            yyyymmdd,
            platform,
            service_detail_id,
            user_id customer_id,
            fare_estimate_id,
            quarter_hour fe_quarter_hour
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            -- AND city = '{city}'
            AND city not in ('Bangalore','Delhi','Hyderabad','Chennai','Pune','Mumbai','Kolkata','Ahmedabad','Jaipur','Chandigarh','Indore','Lucknow','Guwahati','Vijayawada','Coimbatore','Vishakapatnam','Nagpur')
            AND service_name IN ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
            AND platform IN ('android', 'ios')
            
            -- AND user_id IN ('6288c6694281e53bbe33d133', '5a2774b1178fbb6f92482223')
        ),
        
        rr_net_tbl as (
        
        SELECT   
            yyyymmdd,
            service_detail_id,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            quarter_hour rr_quarter_hour
        FROM 
            orders.order_logs_snapshot ols
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            -- AND city_name = '{city}'
            AND city_name not in ('Bangalore','Delhi','Hyderabad','Chennai','Pune','Mumbai','Kolkata','Ahmedabad','Jaipur','Chandigarh','Indore','Lucknow','Guwahati','Vijayawada','Coimbatore','Vishakapatnam','Nagpur')
            AND service_obj_service_name IN ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')

            -- AND customer_id IN ('6288c6694281e53bbe33d133', '5a2774b1178fbb6f92482223')
        ),
        
        merged AS (
        
        SELECT
            fe_tbl.*,
            rr_net_tbl.order_id,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.rr_quarter_hour
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.yyyymmdd = rr_net_tbl.yyyymmdd
            AND fe_tbl.customer_id = rr_net_tbl.customer_id
            AND fe_tbl.service_detail_id = rr_net_tbl.service_detail_id
        )

        SELECT
            platform,
            customer_id,
            COUNT(DISTINCT fare_estimate_id) AS fe_count,
            COUNT(DISTINCT order_id) AS rr_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN fare_estimate_id END) morning_fe,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN fare_estimate_id END) evening_fe,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN order_id END) morning_rr,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN order_id END) evening_rr

        FROM 
            merged
        GROUP BY 1,2
        
"""

df_active_customer_base = pd.read_sql(active_customer_base, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_active_customer_base.head(1)
df_active_customer_base.to_csv(local_extract + 'raw_active_customer_base_{}_{}.csv'.format(refresh_month,city), index=False)

Query 1 running..
Execution time: 6.19 mins


In [6]:
df_active_customer_base = pd.read_csv(local_extract + 'raw_active_customer_base_{}_{}.csv'.format(refresh_month,city))

In [7]:
df_active_customer_base

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr
0,android,66ec3e36d6c48bbb761f8626,15,7,1,0,15,0,7
1,android,6401751ab29bb508a74ad467,14,5,4,7,6,4,1
2,android,61b8f111f3de6f9a984ce37f,19,7,4,13,4,4,2
3,android,5c38a8c04a267149c76cf8a6,3,1,1,3,0,1,0
4,android,652ebcdf9c511543087f33a2,8932,0,0,2612,3908,0,0
...,...,...,...,...,...,...,...,...,...
2687410,android,62b284cb3591be46bb324862,1,1,1,1,0,1,0
2687411,android,66f2697703477d02611f3207,2,0,0,2,0,0,0
2687412,android,646f3b86af2fe1fad4bf323b,1,0,0,0,1,0,0
2687413,android,64b2a7915b5dad28474afbcb,1,0,0,0,1,0,0


In [8]:
df_active_customer_base.fe_count.sum()

18026247

### Segment data

In [9]:
print('Query 2 running..')
start_time = time.time()
customer_segments = f"""

    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_lifetime_rides life_time_rides,
        taxi_lifetime_stage life_time_stage,
        taxi_service_affinity service_affinity
        
    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date =  DATE_FORMAT(DATE_PARSE('{end_date}', '%Y%m%d'), '%Y-%m-%d') -- + INTERVAL '1' DAY 
        AND taxi_recency_segment != 'INACTIVE' 
        -- AND taxi_lifetime_last_ride_city = '{city}'
        AND taxi_lifetime_last_ride_city not in ('Bangalore','Delhi','Hyderabad','Chennai','Pune','Mumbai','Kolkata','Ahmedabad','Jaipur','Chandigarh','Indore','Lucknow','Guwahati','Vijayawada','Coimbatore','Vishakapatnam','Nagpur')

        -- AND customer_id IN ('6288c6694281e53bbe33d133', '5a2774b1178fbb6f92482223')
        
"""
df_customer_segments = pd.read_sql(customer_segments, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_segments.to_csv(local_extract + 'raw_customer_segments_{}_{}.csv'.format(refresh_month,city), index=False)

Query 2 running..
Execution time: 1.15 mins


In [10]:
df_customer_segments = pd.read_csv(local_extract + 'raw_customer_segments_{}_{}.csv'.format(refresh_month,city))

### Age and gender data

In [11]:
print('Query 3 running..')
start_time = time.time()

customer_demo_data = f"""

    WITH fe_customer AS (

        SELECT 
            user_id customer_id
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            -- AND city = '{city}'
            AND city not in ('Bangalore','Delhi','Hyderabad','Chennai','Pune','Mumbai','Kolkata','Ahmedabad','Jaipur','Chandigarh','Indore','Lucknow','Guwahati','Vijayawada','Coimbatore','Vishakapatnam','Nagpur')
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CabPremium', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')

            -- AND user_id IN ('6288c6694281e53bbe33d133', '5a2774b1178fbb6f92482223')
            
        GROUP BY 1
        ),

        demographic_data AS ( 

        SELECT
            customerId customer_id,
            CASE WHEN age BETWEEN 15 AND 60 THEN age ELSE 0 END age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            datasets.customer_single_view
        -- WHERE 
            -- customerId IN ('6288c6694281e53bbe33d133', '5a2774b1178fbb6f92482223')
        )

        SELECT
            a.customer_id,
            COALESCE(age, 0) age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            fe_customer a
        LEFT JOIN 
            demographic_data b
            on a.customer_id = b.customer_id        
"""

df_customer_demo_data = pd.read_sql(customer_demo_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_demo_data.to_csv(local_extract + 'raw_customer_demo_data_{}_{}.csv'.format(refresh_month,city), index=False)

Query 3 running..
Execution time: 1.34 mins


In [12]:
df_customer_demo_data = pd.read_csv(local_extract + 'raw_customer_demo_data_{}_{}.csv'.format(refresh_month,city))

### Use-case data

In [13]:
print('Query 4 running..')
start_time = time.time()

customer_use_case_data = f"""

    WITH use_case AS (

        SELECT
            hex_10, 
            usecase_tag
        FROM
            (
            SELECT 
                hex_10, 
                combined_final_usecase_accuracy as usecase_tag,
                ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
            FROM
                hive.experiments_internal.combined_usecase_hex_10_level
            WHERE 
                aoi = 'Bangalore, India'
            )
        WHERE   
            seq_no = 1
        ),

        active_customers AS (

        SELECT 
            customer_id,
            order_id,
            drop_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'first_half_drop' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON drop_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 8 
            AND cast(substr(quarter_hour,1,2) as real) <= 12    
            -- AND city_name = '{city}'
            AND city_name not in ('Bangalore','Delhi','Hyderabad','Chennai','Pune','Mumbai','Kolkata','Ahmedabad','Jaipur','Chandigarh','Indore','Lucknow','Guwahati','Vijayawada','Coimbatore','Vishakapatnam','Nagpur')
            -- AND customer_id IN ('6288c6694281e53bbe33d133', '5a2774b1178fbb6f92482223')
            
        GROUP BY 1,2,3,4

        UNION 

            SELECT 
            customer_id,
            order_id,
            pickup_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'second_half_pickup' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON pickup_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 13
            AND cast(substr(quarter_hour,1,2) as real) <= 21
            -- AND city_name = '{city}'
            AND city_name not in ('Bangalore','Delhi','Hyderabad','Chennai','Pune','Mumbai','Kolkata','Ahmedabad','Jaipur','Chandigarh','Indore','Lucknow','Guwahati','Vijayawada','Coimbatore','Vishakapatnam','Nagpur')
            -- AND customer_id IN ('6288c6694281e53bbe33d133', '5a2774b1178fbb6f92482223')
        GROUP BY 1,2,3,4

        ),

        merge AS (

        SELECT
            customer_id,
            time_period,
            usecase_tag,
            orders,
            ROW_NUMBER() OVER(PARTITION BY customer_id,time_period ORDER BY orders DESC) seq_no
        FROM 
            (
            SELECT
                customer_id,
                time_period,
                usecase_tag,
                COUNT(DISTINCT order_id) orders
            FROM 
                active_customers
            GROUP BY 1,2,3
            )
        -- WHERE 
        --     LOWER(usecase_tag) != 'unknown'
        )

        SELECT
            customer_id,
            COALESCE(first_half_drop_primary, 'NA') first_half_drop_primary,
            COALESCE(first_half_drop_secondary, 'NA') first_half_drop_secondary,
            COALESCE(second_half_pickup_primary, 'NA') second_half_pickup_primary,
            COALESCE(second_half_pickup_secondary, 'NA') second_half_pickup_secondary
        FROM
        (
        SELECT
            customer_id,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 1 THEN usecase_tag END) first_half_drop_primary,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 2 THEN usecase_tag END) first_half_drop_secondary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 1 THEN usecase_tag END) second_half_pickup_primary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 2 THEN usecase_tag END) second_half_pickup_secondary
        FROM 
            merge 
        WHERE 
            seq_no in (1,2)
        GROUP BY 1
        ) 
"""

df_customer_use_case_data = pd.read_sql(customer_use_case_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_use_case_data.to_csv(local_extract + 'raw_customer_use_case_data_{}_{}.csv'.format(refresh_month,city), index=False)

Query 4 running..
Execution time: 9.32 mins


In [14]:
df_customer_use_case_data = pd.read_csv(local_extract + 'raw_customer_use_case_data_{}_{}.csv'.format(refresh_month,city))

### Appography data

In [18]:
print('Query 5 running..')
start_time = time.time()
customer_appography_query = f"""

    WITH active_customer_base AS (

    SELECT 
        DISTINCT user_id customer_id
    FROM 
        pricing.fare_estimates_enriched 
    WHERE 
        yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
        -- AND city = '{city}'
        AND city not in ('Bangalore','Delhi','Hyderabad','Chennai','Pune','Mumbai','Kolkata','Ahmedabad','Jaipur','Chandigarh','Indore','Lucknow','Guwahati','Vijayawada','Coimbatore','Vishakapatnam','Nagpur')
        AND service_name in ('Auto', 'Link')
        -- IN ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND platform = 'android'
    ),

    appographydata AS (
    
    SELECT
        userid customer_id, 
        appographydata AS app_lists
    FROM 
        datasets.customer_apps_installed_snapshot
    )
    
    SELECT
        appographydata.*
    FROM 
        active_customer_base
    JOIN 
        appographydata
        ON appographydata.customer_id = active_customer_base.customer_id
        
"""
df_customer_appography_data = pd.read_sql(customer_appography_query, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_appography_data.to_csv(local_extract + 'raw_customer_appography_data_{}_{}.csv'.format(refresh_month,city), index=False)

Query 5 running..
Execution time: 5.92 mins


In [19]:
df_customer_appography_data = pd.read_csv(local_extract + 'raw_customer_appography_data_{}_{}.csv'.format(refresh_month,city))

In [20]:
start_time = time.time()

#regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r'"([^"]*)"', text)]

df_customer_appography_data['app_lists'] = df_customer_appography_data['app_lists'].apply(lambda x: extract_and_trim(x))

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time : ", execution_time, " mins")

Execution time :  0.5  mins


### Category Mapping

In [21]:
print('Query 6 running..')
start_time = time.time()
category_mapping = f"""

    SELECT 
        lower(app_name) app_name, 
        lower(category) category 
    FROM 
        experiments_internal.customer_app_category_mapping
        
"""
category_list = pd.read_sql(category_mapping, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
category_list.to_csv(local_extract + 'category_mapping_{}_{}.csv'.format(refresh_month,city), index=False)

Query 6 running..
Execution time: 0.04 mins


### Merge

In [22]:
def merging_df(df1, df2, df3, df4, df5):

    start_time = time.time()

    df_active_customer_base = df1
    df_customer_segments = df2
    df_customer_demo_data = df3
    df_customer_use_case_data = df4
    df_customer_appography_data = df5

    print(df_active_customer_base.columns)
    print(df_customer_segments.columns)
    print(df_customer_demo_data.columns)
    print(df_customer_use_case_data.columns)
    print(df_customer_appography_data.columns)
    
    print('Merging 1.. ')
    merge1 = pd.merge(df_active_customer_base,df_customer_segments,how='left',on = ['customer_id'])
    
    print('Merging 2..')
    merge2 = pd.merge(merge1,df_customer_demo_data,how='left',on = ['customer_id'])
    
    print('Merging 3..')
    merge3 = pd.merge(merge2,df_customer_use_case_data,how='left',on = ['customer_id'])
    
    print('Merging 4..')
    merge4 = pd.merge(merge3,df_customer_appography_data,how='left',on = ['customer_id'])
    
    df_all_customer = merge4
    
    del merge1
    del merge2
    del merge3
    del merge4
    
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")

    df_all_customer.to_csv(local_extract + 'processed_all_customer_{}_{}.csv'.format(refresh_month,city), index=False)

    return df_all_customer

In [23]:
main_df_all_customer = merging_df(df_active_customer_base, 
                             df_customer_segments, 
                             df_customer_demo_data, 
                             df_customer_use_case_data, 
                             df_customer_appography_data)

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr'],
      dtype='object')
Index(['customer_id', 'ltr_segment', 'life_time_rides', 'life_time_stage',
       'service_affinity'],
      dtype='object')
Index(['customer_id', 'age', 'gender'], dtype='object')
Index(['customer_id', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary'],
      dtype='object')
Index(['customer_id', 'app_lists'], dtype='object')
Merging 1.. 
Merging 2..
Merging 3..
Merging 4..
Execution time: 0.33 mins


In [24]:
# Calculate the split indices
split_size = len(main_df_all_customer) // 3
split_size

895805

In [27]:
# Split the DataFrame into 3 parts
df1 = main_df_all_customer.iloc[:split_size]
df2 = main_df_all_customer.iloc[split_size:2*split_size]
df3 = main_df_all_customer.iloc[2*split_size:]

In [122]:
df_all_customer = df3.copy()

In [123]:
df_all_customer

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_lists
1791610,android,64392f269e048b96f579cfea,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,19,MALE,NaN,NaN,NaN,NaN,"[AmazonPrimeVideo, Zomato, AxisMobile, BookMyS..."
1791611,android,63d0e3730891fc02e0a2e5c6,4,1,0,3,1,1,0,PHH,26.0,DORMANT,ONLY_AUTO,0,FEMALE,Unknown,NaN,NaN,NaN,"[BigBasket, BookMyShow, Domino'sPizza, Faceboo..."
1791612,android,658ada47d9b0e78ccf0ecdf2,16,3,3,8,6,2,1,HH,3.0,HANDHOLDING,UNKNOWN,0,MALE,Unknown,NaN,Unknown,NaN,"[AmazonPrimeVideo, Zomato, ALTBalaji, BookMySh..."
1791613,android,61fb8daeebef8269e82c4036,6,3,2,2,2,0,2,PHH,42.0,COMMITTED,ONLY_LINK,27,MALE,NaN,NaN,Unknown,NaN,"[AdobeScan, AmazonPrimeVideo, Apollo247, Zomat..."
1791614,android,6290c9e2e9e87c285972a725,3,3,3,2,0,2,0,PHH,123.0,COMMITTED,NO_AFFINITY,0,FEMALE,Unknown,NaN,NaN,NaN,"[AmazonPrimeVideo, Zomato, Facebook, Messenger..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2687410,android,62b284cb3591be46bb324862,1,1,1,1,0,1,0,PHH,37.0,CHURN_OTB,LINK_AUTO,30,FEMALE,Unknown,NaN,NaN,NaN,"[Aha, Brainly, AmazonPrimeVideo, Apollo247, Zo..."
2687411,android,66f2697703477d02611f3207,2,0,0,2,0,0,0,NaN,NaN,NaN,NaN,0,MALE,NaN,NaN,NaN,NaN,"[AdobeLightroom, AmazonPrimeVideo, AmazonMusic..."
2687412,android,646f3b86af2fe1fad4bf323b,1,0,0,0,1,0,0,NaN,NaN,NaN,NaN,0,MALE,NaN,NaN,NaN,NaN,"[AdobeScan, GoogleAuthenticator, GoogleDrive, ..."
2687413,android,64b2a7915b5dad28474afbcb,1,0,0,0,1,0,0,NaN,NaN,NaN,NaN,0,MALE,NaN,NaN,NaN,NaN,"[Adobe Scan, Airtel Thanks, Amazon Prime Video..."


### One-Hot-Encoder

In [124]:
categorical_columns = ['gender', 'ltr_segment', 'life_time_stage', 'service_affinity', 
                       'first_half_drop_primary', 'first_half_drop_secondary',
                       'second_half_pickup_primary', 'second_half_pickup_secondary']

def one_hot_encoder(df):
    df_categorical = df[categorical_columns].copy()
    
    for column_name in categorical_columns:
        df_categorical[column_name] = df_categorical[column_name].fillna(column_name + '_unknown')
    
    # Initialize the OneHotEncoder
    encoder = OneHotEncoder()
    
    # Fit and transform the categorical columns
    encoded_categorical = encoder.fit_transform(df_categorical)
    
    # Convert the encoded result into a DataFrame
    encoded_df = pd.DataFrame(encoded_categorical.toarray(), 
                              columns=encoder.get_feature_names_out(input_features=categorical_columns))
    
    # Now drop the original categorical columns from the original DataFrame
    df = df.drop(columns=categorical_columns)

    # Concatenate the original DataFrame with the encoded DataFrame
    df = pd.concat([df, encoded_df], axis=1)
    
    
    return df

In [125]:
df_one_hot_encoded = one_hot_encoder(df_all_customer)
print(df_one_hot_encoded.shape)
df_one_hot_encoded['life_time_rides'].fillna(0, inplace=True)
df_one_hot_encoded.head(3)

(1791610, 45)


,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,life_time_rides,age,app_lists,gender_FEMALE,gender_MALE,gender_OTHERS,gender_UNKNOWN,ltr_segment_HH,ltr_segment_PHH,ltr_segment_UNKNOWN,ltr_segment_ltr_segment_unknown,life_time_stage_CHURN_OTB,life_time_stage_COMMITTED,life_time_stage_DETOX,life_time_stage_DORMANT,life_time_stage_HANDHOLDING,life_time_stage_HOOK,life_time_stage_SOFT_CHURN,life_time_stage_SUSTENANCE,life_time_stage_UNKNOWN,life_time_stage_life_time_stage_unknown,service_affinity_AUTO_CAB,service_affinity_LINK_AUTO,service_affinity_LINK_CAB,service_affinity_NO_AFFINITY,service_affinity_ONLY_AUTO,service_affinity_ONLY_CAB,service_affinity_ONLY_LINK,service_affinity_UNKNOWN,service_affinity_service_affinity_unknown,first_half_drop_primary_Unknown,first_half_drop_primary_first_half_drop_primary_unknown,first_half_drop_secondary_first_half_drop_secondary_unknown,second_half_pickup_primary_Unknown,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_secondary_second_half_pickup_secondary_unknown
1791610,android,64392f269e048b96f579cfea,6.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,19.0,"[AmazonPrimeVideo, Zomato, AxisMobile, BookMyS...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1791611,android,63d0e3730891fc02e0a2e5c6,4.0,1.0,0.0,3.0,1.0,1.0,0.0,26.0,0.0,"[BigBasket, BookMyShow, Domino'sPizza, Faceboo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1791612,android,658ada47d9b0e78ccf0ecdf2,16.0,3.0,3.0,8.0,6.0,2.0,1.0,3.0,0.0,"[AmazonPrimeVideo, Zomato, ALTBalaji, BookMySh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [126]:
list_encoded_columns = df_one_hot_encoded.columns.tolist()
exclude_list = ['platform', 'fe_count', 'rr_count', 'net_count', 
                'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 
                'life_time_rides', 'age', 'app_lists', 
                'gender_FEMALE', 'gender_MALE', 'gender_OTHERS', 'gender_UNKNOWN', 'gender_gender_unknown']

encoded_columns = [item for item in list_encoded_columns if item not in exclude_list]
encoded_columns

['customer_id',
 'ltr_segment_HH',
 'ltr_segment_PHH',
 'ltr_segment_UNKNOWN',
 'ltr_segment_ltr_segment_unknown',
 'life_time_stage_CHURN_OTB',
 'life_time_stage_COMMITTED',
 'life_time_stage_DETOX',
 'life_time_stage_DORMANT',
 'life_time_stage_HANDHOLDING',
 'life_time_stage_HOOK',
 'life_time_stage_SOFT_CHURN',
 'life_time_stage_SUSTENANCE',
 'life_time_stage_UNKNOWN',
 'life_time_stage_life_time_stage_unknown',
 'service_affinity_AUTO_CAB',
 'service_affinity_LINK_AUTO',
 'service_affinity_LINK_CAB',
 'service_affinity_NO_AFFINITY',
 'service_affinity_ONLY_AUTO',
 'service_affinity_ONLY_CAB',
 'service_affinity_ONLY_LINK',
 'service_affinity_UNKNOWN',
 'service_affinity_service_affinity_unknown',
 'first_half_drop_primary_Unknown',
 'first_half_drop_primary_first_half_drop_primary_unknown',
 'first_half_drop_secondary_first_half_drop_secondary_unknown',
 'second_half_pickup_primary_Unknown',
 'second_half_pickup_primary_second_half_pickup_primary_unknown',
 'second_half_pickup_sec

In [127]:
df_one_hot_encoded = df_one_hot_encoded[encoded_columns]

## Appography & Use-Case

In [128]:
df_all_customer.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 'ltr_segment',
       'life_time_rides', 'life_time_stage', 'service_affinity', 'age',
       'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary',
       'app_lists'],
      dtype='object')

In [129]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_lists'].isna() == True, 'NA', 'AVAILABLE')

In [130]:
df_all_customer.appo_presence.unique()

array(['AVAILABLE', 'NA'], dtype=object)

In [131]:
df_triangulation_1 = df_all_customer[(df_all_customer['platform'].isin(['android']) 
                                      & 
                                      df_all_customer['appo_presence'].isin(['AVAILABLE']))]\
                                    [['customer_id', 'age', 'appo_presence','app_lists',
                                      'fe_count', 'rr_count', 'net_count',
                                      'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 
                                      'ltr_segment', 'life_time_rides', 'life_time_stage', 'service_affinity',
                                      'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
                                      'second_half_pickup_primary', 'second_half_pickup_secondary',
                                     ]]

df_triangulation_1.head(2)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
1791610,64392f269e048b96f579cfea,19,AVAILABLE,"[AmazonPrimeVideo, Zomato, AxisMobile, BookMyS...",6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN
1791611,63d0e3730891fc02e0a2e5c6,0,AVAILABLE,"[BigBasket, BookMyShow, Domino'sPizza, Faceboo...",4,1,0,3,1,1,0,PHH,26.0,DORMANT,ONLY_AUTO,FEMALE,Unknown,NaN,NaN,NaN


In [132]:
df_triangulation_1.customer_id.nunique()

777953

### Explode

In [133]:
## Explode list into individual rows
df_main_apps_explode = df_triangulation_1.explode('app_lists')
df_main_apps_explode['app_lists'] = df_main_apps_explode['app_lists'].str.lower()

df_main_apps_explode.head(4)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
1791610,64392f269e048b96f579cfea,19,AVAILABLE,amazonprimevideo,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN
1791610,64392f269e048b96f579cfea,19,AVAILABLE,zomato,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN
1791610,64392f269e048b96f579cfea,19,AVAILABLE,axismobile,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN
1791610,64392f269e048b96f579cfea,19,AVAILABLE,bookmyshow,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN


### Merge - category mapping

In [134]:
df_main_apps_explode.head(2)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
1791610,64392f269e048b96f579cfea,19,AVAILABLE,amazonprimevideo,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN
1791610,64392f269e048b96f579cfea,19,AVAILABLE,zomato,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN


In [135]:
df_main_app_category =  pd.merge(df_main_apps_explode, 
                                 category_list[['app_name', 'category']], 
                                 how='left', 
                                 left_on='app_lists', 
                                 right_on='app_name'
                              )
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,category
0,64392f269e048b96f579cfea,19,AVAILABLE,amazonprimevideo,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,NaN,NaN
1,64392f269e048b96f579cfea,19,AVAILABLE,zomato,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,zomato,delivery_food
2,64392f269e048b96f579cfea,19,AVAILABLE,zomato,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,zomato,delivery_food
3,64392f269e048b96f579cfea,19,AVAILABLE,axismobile,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,NaN,NaN


## Feature engineering 

### Transform Categories

In [136]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_lists,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,category
0,64392f269e048b96f579cfea,19,AVAILABLE,amazonprimevideo,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,NaN,NaN
1,64392f269e048b96f579cfea,19,AVAILABLE,zomato,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,zomato,delivery_food
2,64392f269e048b96f579cfea,19,AVAILABLE,zomato,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,zomato,delivery_food
3,64392f269e048b96f579cfea,19,AVAILABLE,axismobile,6,0,0,4,2,0,0,NaN,NaN,NaN,NaN,MALE,NaN,NaN,NaN,NaN,NaN,NaN


In [137]:
df_main_app_category[df_main_app_category['app_lists'].isna()].nunique()

customer_id                     0
age                             0
appo_presence                   0
app_lists                       0
fe_count                        0
rr_count                        0
net_count                       0
morning_fe                      0
evening_fe                      0
morning_rr                      0
evening_rr                      0
ltr_segment                     0
life_time_rides                 0
life_time_stage                 0
service_affinity                0
gender                          0
first_half_drop_primary         0
first_half_drop_secondary       0
second_half_pickup_primary      0
second_half_pickup_secondary    0
app_name                        0
category                        0
dtype: int64

In [138]:
most_common_apps = ['google maps', 'gmail', 'youtube', 'whatsapp', 'clock', 'calculator', 'google calendar', 'google photos']

In [139]:
df_customer_mapped_v1 = df_main_app_category[~df_main_app_category['app_lists'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_lists', lambda x: list(set(x))),
                             category = ('category', lambda x: list(set(x)))
                            )\
                        .reset_index()

df_customer_mapped_v1.head(4)

,customer_id,age,app_list,category
0,573f28f39b0ffc283676f251,37,"[paytm money, uber, irctc rail connect, telegr...","[travel_ridehailing, tools, travel_bookings, d..."
1,573f28f39b0ffc283676f2ba,32,"[youtubemusic, hotstar, naukri, paytm, truecal...","[travel_bookings, tools, ecommerce, delivery_f..."
2,573f28f49b0ffc283676f57d,0,"[youtubemusic, hotstar, pinterest, paytm, lens...","[travel_bookings, travel_ridehailing, tools, d..."
3,573f28f49b0ffc283676f646,37,"[cleartrip, hotstar, pinterest, paytm, purplle...","[travel_bookings, travel_ridehailing, tools, d..."


In [140]:
unique_categories = set(category for sublist in df_customer_mapped_v1['app_list'] for category in sublist)
unique_categories_list = list(unique_categories)
print("Count of unique apps:", len(unique_categories))
# sorted(unique_categories)

Count of unique apps: 605


In [141]:
df_customer_mapped_v1.head(2)

,customer_id,age,app_list,category
0,573f28f39b0ffc283676f251,37,"[paytm money, uber, irctc rail connect, telegr...","[travel_ridehailing, tools, travel_bookings, d..."
1,573f28f39b0ffc283676f2ba,32,"[youtubemusic, hotstar, naukri, paytm, truecal...","[travel_bookings, tools, ecommerce, delivery_f..."


In [142]:
df_customer_mapped_v1.shape

(777928, 4)

In [143]:
for category in unique_categories:
    df_customer_mapped_v1[category] = df_customer_mapped_v1['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v1.head(1)

,customer_id,age,app_list,category,mi fit,whitehat jr,skype,paytm,file commander,cars24,amazonalexa,ndtv news,dropbox,youcammakeup,googlechat,carrom pool,espncricinfo,vidyakul,jira,bansal classes,zoom,airtel thanks,phonepe,urbancompany,economictimes,yatra,icicidirect: stocks,upstox,fiton,toppr,fresh to home,air india,my study life,vedantu,bhim upi,sunnxt,physicswallah,booking.com,free fire max,home workout - no equipment,kotak neo,business standard,noisefit,medplus mart,mojlite,temple run 2,physics wallah,premierleague,mumbai metro,big bazaar,...,pubg mobile,line,linkedin learning,hungamamusic,myntra,ncert books,quickride,lenskart,indigo,abp live,tripadvisor,medium,mcdonald's india,tumblr,cricketleague,geeks for geeks,sadhguru,ms edge,soundcloud,ajio,googlesheets,caclubindia,groupme,axis mobile,groww,firstcry,msauthenticator,freelancer,webex,mobvoi,uts,amazonminitv,coffee meets bagel,bansalclasses,angel broking,nobroker,carrompool,jiomart,shiksha mitra,mini militia,kite by zerodha,hungama music,messenger,kfc india,freshtohome,signal,theindianexpress,vision11,photomath,irctcrailconnect
0,573f28f39b0ffc283676f251,37,"[paytm money, uber, irctc rail connect, telegr...","[travel_ridehailing, tools, travel_bookings, d...",0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Bins definitions

In [144]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['Below-21', '21-25',  '26-30', '31-35', '36-45', 'Above-45']

df_customer_mapped_v1['age_group'] = pd.cut(df_customer_mapped_v1['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [145]:
main_list = ['customer_id', 'age', 'age_group' ,'category']
unique_column_list = main_list + unique_categories_list

In [146]:
df_customer_mapped_v1 = df_customer_mapped_v1[unique_column_list]
df_customer_mapped_v1.head(1)

,customer_id,age,age_group,category,mi fit,whitehat jr,skype,paytm,file commander,cars24,amazonalexa,ndtv news,dropbox,youcammakeup,googlechat,carrom pool,espncricinfo,vidyakul,jira,bansal classes,zoom,airtel thanks,phonepe,urbancompany,economictimes,yatra,icicidirect: stocks,upstox,fiton,toppr,fresh to home,air india,my study life,vedantu,bhim upi,sunnxt,physicswallah,booking.com,free fire max,home workout - no equipment,kotak neo,business standard,noisefit,medplus mart,mojlite,temple run 2,physics wallah,premierleague,mumbai metro,big bazaar,...,pubg mobile,line,linkedin learning,hungamamusic,myntra,ncert books,quickride,lenskart,indigo,abp live,tripadvisor,medium,mcdonald's india,tumblr,cricketleague,geeks for geeks,sadhguru,ms edge,soundcloud,ajio,googlesheets,caclubindia,groupme,axis mobile,groww,firstcry,msauthenticator,freelancer,webex,mobvoi,uts,amazonminitv,coffee meets bagel,bansalclasses,angel broking,nobroker,carrompool,jiomart,shiksha mitra,mini militia,kite by zerodha,hungama music,messenger,kfc india,freshtohome,signal,theindianexpress,vision11,photomath,irctcrailconnect
0,573f28f39b0ffc283676f251,37,36-45,"[travel_ridehailing, tools, travel_bookings, d...",0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Appo and one-hot-encoding 

In [147]:
# Adding required additional info by one-hot-encoding 

df_customer_mapped_v1 = pd.merge(df_customer_mapped_v1, 
                                 df_one_hot_encoded,
                                 how='left',
                                 left_on='customer_id',
                                 right_on='customer_id'
                                )
# df_customer_mapped_v1['life_time_rides'].fillna(0, inplace=True)

df_customer_mapped_v1.head(1)

,customer_id,age,age_group,category,mi fit,whitehat jr,skype,paytm,file commander,cars24,amazonalexa,ndtv news,dropbox,youcammakeup,googlechat,carrom pool,espncricinfo,vidyakul,jira,bansal classes,zoom,airtel thanks,phonepe,urbancompany,economictimes,yatra,icicidirect: stocks,upstox,fiton,toppr,fresh to home,air india,my study life,vedantu,bhim upi,sunnxt,physicswallah,booking.com,free fire max,home workout - no equipment,kotak neo,business standard,noisefit,medplus mart,mojlite,temple run 2,physics wallah,premierleague,mumbai metro,big bazaar,...,mobvoi,uts,amazonminitv,coffee meets bagel,bansalclasses,angel broking,nobroker,carrompool,jiomart,shiksha mitra,mini militia,kite by zerodha,hungama music,messenger,kfc india,freshtohome,signal,theindianexpress,vision11,photomath,irctcrailconnect,ltr_segment_HH,ltr_segment_PHH,ltr_segment_UNKNOWN,ltr_segment_ltr_segment_unknown,life_time_stage_CHURN_OTB,life_time_stage_COMMITTED,life_time_stage_DETOX,life_time_stage_DORMANT,life_time_stage_HANDHOLDING,life_time_stage_HOOK,life_time_stage_SOFT_CHURN,life_time_stage_SUSTENANCE,life_time_stage_UNKNOWN,life_time_stage_life_time_stage_unknown,service_affinity_AUTO_CAB,service_affinity_LINK_AUTO,service_affinity_LINK_CAB,service_affinity_NO_AFFINITY,service_affinity_ONLY_AUTO,service_affinity_ONLY_CAB,service_affinity_ONLY_LINK,service_affinity_UNKNOWN,service_affinity_service_affinity_unknown,first_half_drop_primary_Unknown,first_half_drop_primary_first_half_drop_primary_unknown,first_half_drop_secondary_first_half_drop_secondary_unknown,second_half_pickup_primary_Unknown,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_secondary_second_half_pickup_secondary_unknown
0,573f28f39b0ffc283676f251,37,36-45,"[travel_ridehailing, tools, travel_bookings, d...",0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [148]:
# Find columns with NaN values
columns_with_nan = df_customer_mapped_v1.columns[df_customer_mapped_v1.isna().any()].tolist()

# Filter DataFrame to get only columns with NaN values
df_with_nan = df_customer_mapped_v1[columns_with_nan]
df_with_nan.isnull().sum()

age_group                                                            523689
ltr_segment_HH                                                       778178
ltr_segment_PHH                                                      778178
ltr_segment_UNKNOWN                                                  778178
ltr_segment_ltr_segment_unknown                                      778178
life_time_stage_CHURN_OTB                                            778178
life_time_stage_COMMITTED                                            778178
life_time_stage_DETOX                                                778178
life_time_stage_DORMANT                                              778178
life_time_stage_HANDHOLDING                                          778178
life_time_stage_HOOK                                                 778178
life_time_stage_SOFT_CHURN                                           778178
life_time_stage_SUSTENANCE                                           778178
life_time_st

In [149]:
df_with_nan.isnull()

,age_group,ltr_segment_HH,ltr_segment_PHH,ltr_segment_UNKNOWN,ltr_segment_ltr_segment_unknown,life_time_stage_CHURN_OTB,life_time_stage_COMMITTED,life_time_stage_DETOX,life_time_stage_DORMANT,life_time_stage_HANDHOLDING,life_time_stage_HOOK,life_time_stage_SOFT_CHURN,life_time_stage_SUSTENANCE,life_time_stage_UNKNOWN,life_time_stage_life_time_stage_unknown,service_affinity_AUTO_CAB,service_affinity_LINK_AUTO,service_affinity_LINK_CAB,service_affinity_NO_AFFINITY,service_affinity_ONLY_AUTO,service_affinity_ONLY_CAB,service_affinity_ONLY_LINK,service_affinity_UNKNOWN,service_affinity_service_affinity_unknown,first_half_drop_primary_Unknown,first_half_drop_primary_first_half_drop_primary_unknown,first_half_drop_secondary_first_half_drop_secondary_unknown,second_half_pickup_primary_Unknown,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_secondary_second_half_pickup_secondary_unknown
0,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
1,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
3,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
4,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
778173,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
778174,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
778175,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
778176,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


## Stratified Sampling & Model

In [150]:
df_customer_mapped_v1_age_w = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()]
df_customer_mapped_v1_age_na = df_customer_mapped_v1[df_customer_mapped_v1['age_group'].isna()]

# Keep a copy of the columns to be retained
df_customer_mapped_v1_age_na_columns = df_customer_mapped_v1_age_na[['customer_id', 'category', 'age']].copy()

# Drop unnecessary columns for model training
df_customer_mapped_v1_age_w.drop(columns=['customer_id', 'category', 'age'], axis=1, inplace=True)
df_customer_mapped_v1_age_na.drop(columns=['customer_id', 'category', 'age'], axis=1, inplace=True)

In [151]:
def evaluate_target_vs_rest(df, target_column):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    classifiers = {}
    
    for target_class in target_classes:
        
        # Create a copy of df to avoid modifying the original dataframe
        df_temp = df.copy()
        
        # Split the bins target_class vs Rest 
        df_target = df_temp[df_temp[target_column] == target_class]
        df_rest = df_temp[df_temp[target_column] != target_class]
        
        df_other_sample = df_rest.sample(n=df_target.shape[0], random_state=1)
        df_combined = pd.concat([df_target, df_other_sample], ignore_index=True)
        
        # Create a binary target column where target_class is 1 and rest are 0
        df_combined['target'] = df_combined[target_column].apply(lambda x: 1 if x == target_class else 0)

        # Ensure all column names are strings
        df_combined.columns = df_combined.columns.astype(str)

        # train_test_split
        X_train, X_test, y_train, y_test = train_test_split(df_combined.drop(columns=[target_column, 'target']), 
                                                            df_combined['target'], 
                                                            test_size=0.2, 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        classifiers[target_class] = rf_model

        # Extract feature importances
        feature_importances = rf_model.feature_importances_

        # Get the indices of top features
        top_feature_indices = np.argsort(feature_importances)[::-1][:20] # Adjust 20 as needed
        # Get the names of top features
        top_features = X_train.columns[top_feature_indices]

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest:")
        print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        print("Precision : ", round(precision * 100.00, 2), "%")
        print("Recall : ", round(recall * 100.00, 2), "%")
        print("F1 Score : ", round(f1 * 100.00, 2), "%")
        print("Top Features:")
        for feature in top_features:
            print(feature)
        print("---------------------------------------")
        
    return classifiers

In [152]:
classifiers = evaluate_target_vs_rest(df_customer_mapped_v1_age_w, 'age_group')
classifiers

Results for class 21-25 vs Rest:
Accuracy :  71.44 %
Precision :  71.14 %
Recall :  73.22 %
F1 Score :  72.16 %
Top Features:
snapchat
telegram
messenger
facebook
cred
spotify
pubgmobile
paytm
youtubemusic
truecaller
mxplayer
whereismytrain
instagram
digilocker
imobilepay
flipkart
linkedin
hotstar
snapseed
googlecalendar
---------------------------------------
Results for class 26-30 vs Rest:
Accuracy :  60.74 %
Precision :  60.19 %
Recall :  61.82 %
F1 Score :  60.99 %
Top Features:
snapchat
truecaller
messenger
youtubemusic
telegram
netflix
paytm
ola
whereismytrain
spotify
meesho
hotstar
facebook
swiggy
flipkart
linkedin
googlehome
googlecalendar
zomato
sbiyono
---------------------------------------
Results for class 31-35 vs Rest:
Accuracy :  64.19 %
Precision :  62.53 %
Recall :  70.34 %
F1 Score :  66.21 %
Top Features:
snapchat
messenger
telegram
paytm
flipkart
youtubemusic
truecaller
whereismytrain
facebook
linkedin
mxplayer
meesho
zomato
spotify
swiggy
cred
googlecalendar
netf

{'21-25': RandomForestClassifier(random_state=42),
 '26-30': RandomForestClassifier(random_state=42),
 '31-35': RandomForestClassifier(random_state=42),
 '36-45': RandomForestClassifier(random_state=42),
 'Above-45': RandomForestClassifier(random_state=42),
 'Below-21': RandomForestClassifier(random_state=42)}

In [153]:
def predict_age_group(df, classifiers):
    predictions = pd.DataFrame(index=df.index)
    
    for target_class, classifier in classifiers.items():
        # Ensure the features used in prediction match the training data
        X = df[[col for col in df.columns if col != 'age_group']]
        predictions[f'{target_class}'] = classifier.predict_proba(X)[:, 1]
        
    binary_predictions = (predictions > 0.5).astype(int)
    
    return binary_predictions

In [154]:
# Use the models to predict the age group for df_customer_mapped_v1_age_na
predictions = predict_age_group(df_customer_mapped_v1_age_na, classifiers)
predictions

,21-25,26-30,31-35,36-45,Above-45,Below-21
2,0,0,1,1,1,0
5,0,0,0,1,1,1
9,0,1,1,0,0,0
12,0,1,1,1,0,0
20,0,0,1,1,0,0
...,...,...,...,...,...,...
778173,1,1,0,0,0,0
778174,0,1,0,0,1,0
778175,1,1,0,0,0,0
778176,0,0,1,0,0,1


In [155]:
predictions['total_sum'] = predictions.sum(axis=1)

# Merge the original columns with the prediction results
df_customer_mapped_v1_age_na_with_predictions = pd.concat([df_customer_mapped_v1_age_na_columns['customer_id'], predictions], axis=1)
df_customer_mapped_v1_age_na_with_predictions.head(2)

,customer_id,21-25,26-30,31-35,36-45,Above-45,Below-21,total_sum
2,573f28f49b0ffc283676f57d,0,0,1,1,1,0,3
5,573f28f59b0ffc283676fe7d,0,0,0,1,1,1,3


In [156]:
df_customer_mapped_v1_age_na_with_predictions.total_sum.value_counts(normalize=True)

total_sum
2    0.503102
3    0.321173
1    0.103537
4    0.057891
0    0.009572
5    0.004535
6    0.000189
Name: proportion, dtype: float64

### Final Customer Age Class

In [157]:
# Define a function to apply the logic
def age_weight_function(row):
    cols_with_one = [col for col in ['Below-21', '21-25', '26-30', '31-35', '36-45', 'Above-45'] if row[col] == 1]
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return cols_with_one[0]
    elif total_sum == 2:
        return cols_with_one[-1]
    elif total_sum == 3 or total_sum == 5:
        return cols_with_one[len(cols_with_one) // 2]
    elif total_sum == 4:
        return cols_with_one[2]
    elif total_sum == 6:
        return cols_with_one[4]
    else:
        return 'UNKNOWN'

def get_confidence_tag(row):
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return 'VERY_HIGH'
    elif total_sum == 2:
        return 'HIGH'
    elif total_sum == 3:
        return 'MEDIUM'
    elif total_sum in [4, 5, 6]:
        return 'LOW'
    else:
        return 'UNKNOWN'

In [158]:
df_customer_mapped_v1_age_na_with_predictions['age_group'] = df_customer_mapped_v1_age_na_with_predictions.apply(age_weight_function, axis=1)
df_customer_mapped_v1_age_na_with_predictions['confidence_tag'] = df_customer_mapped_v1_age_na_with_predictions.apply(get_confidence_tag, axis=1)

In [159]:
df_customer_mapped_v1_age_na_with_predictions

,customer_id,21-25,26-30,31-35,36-45,Above-45,Below-21,total_sum,age_group,confidence_tag
2,573f28f49b0ffc283676f57d,0,0,1,1,1,0,3,36-45,MEDIUM
5,573f28f59b0ffc283676fe7d,0,0,0,1,1,1,3,36-45,MEDIUM
9,573f28f69b0ffc28367701a6,0,1,1,0,0,0,2,31-35,HIGH
12,573f28f79b0ffc28367708a3,0,1,1,1,0,0,3,31-35,MEDIUM
20,573f28fe9b0ffc28367719ff,0,0,1,1,0,0,2,36-45,HIGH
...,...,...,...,...,...,...,...,...,...,...
778173,66faeb924fb46fe38a0a0884,1,1,0,0,0,0,2,26-30,HIGH
778174,66faebc4e25dcc5b5571bcfd,0,1,0,0,1,0,2,Above-45,HIGH
778175,66faec19e25dcc75b871ca46,1,1,0,0,0,0,2,26-30,HIGH
778176,66faeca3bab56e05a3370592,0,0,1,0,0,1,2,31-35,HIGH


In [160]:
df_customer_mapped_v1_age_na_with_predictions.age_group.value_counts(normalize=True)

age_group
21-25       0.387108
36-45       0.245040
26-30       0.140688
31-35       0.138611
Above-45    0.058789
Below-21    0.020191
UNKNOWN     0.009572
Name: proportion, dtype: float64

In [161]:
df_customer_mapped_v1_age_na_with_predictions.confidence_tag.value_counts(normalize=True)

confidence_tag
HIGH         0.503102
MEDIUM       0.321173
VERY_HIGH    0.103537
LOW          0.062615
UNKNOWN      0.009572
Name: proportion, dtype: float64

In [162]:
df_final1 = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()][['customer_id', 'age_group']]
df_final1['confidence_tag'] = 'VERY_HIGH'

df_final2 = df_customer_mapped_v1_age_na_with_predictions[['customer_id', 'age_group', 'confidence_tag']]

df_predicated_age_bins = pd.concat([df_final1,df_final2])
df_predicated_age_bins.shape

(778178, 3)

In [163]:
df_predicated_age_bins.confidence_tag.value_counts(normalize=True)

confidence_tag
VERY_HIGH    0.396709
HIGH         0.338572
MEDIUM       0.216139
LOW          0.042138
UNKNOWN      0.006442
Name: proportion, dtype: float64

In [164]:
df_predicated_age_bins['run_date'] = '2024-10-01'

In [165]:
df_predicated_age_bins.to_csv( local_extract + 'final/{}_customer_predicted_age_oct2024_3.csv'.format(city), index=False)

In [166]:
df_predicated_age_bins.head(50)

,customer_id,age_group,confidence_tag,run_date
0,573f28f39b0ffc283676f251,36-45,VERY_HIGH,2024-10-01
1,573f28f39b0ffc283676f2ba,31-35,VERY_HIGH,2024-10-01
3,573f28f49b0ffc283676f646,36-45,VERY_HIGH,2024-10-01
4,573f28f59b0ffc283676fe2e,36-45,VERY_HIGH,2024-10-01
6,573f28f69b0ffc283676ffa7,31-35,VERY_HIGH,2024-10-01
7,573f28f69b0ffc283677010b,26-30,VERY_HIGH,2024-10-01
8,573f28f69b0ffc283677012f,31-35,VERY_HIGH,2024-10-01
10,573f28f69b0ffc283677039b,31-35,VERY_HIGH,2024-10-01
11,573f28f79b0ffc2836770716,36-45,VERY_HIGH,2024-10-01
13,573f28f89b0ffc2836770dba,36-45,VERY_HIGH,2024-10-01
